# Register agents in entitycore

In [2]:
import pandas as pd
import os

from datetime import datetime
from entitysdk import Client, ProjectContext, models
from obi_auth import get_token

In [3]:
token = get_token(environment="staging")
project_context = ProjectContext.from_vlab_url("https://staging.openbraininstitute.org/app/virtual-lab/lab/1f91f30e-1489-4e2a-8eb7-1217257c8e19/project/7a411785-6895-4839-aaa2-d9f76e09875a/home")
client = Client(environment="staging", project_context=project_context, token_manager=token)

# token = get_token(environment="production")
# project_context = ProjectContext.from_vlab_url("https://www.openbraininstitute.org/app/virtual-lab/lab/5f8376bf-b84f-4188-8ef5-e1df3d7529b4/project/7d22829c-edc6-4b1d-8ab9-99dd9e511e74/home")
# client = Client(environment="production", project_context=project_context, token_manager=token)

In [4]:
agents_root = "/Users/pokorny/OneDrive - Open Brain Institute/Circuits hardcoded/entity_core_agents"

### Load list of agents to be registered

In [5]:
person_file = os.path.join(agents_root, "person_Allen-V1.csv")
organization_file = os.path.join(agents_root, "organization_Allen-V1.csv")
consortium_file = os.path.join(agents_root, "consortium_Allen-V1.csv")

persons = pd.read_csv(person_file)
organizations = pd.read_csv(organization_file)
consortiums = pd.read_csv(consortium_file)
persons["ID"] = "-"
organizations["ID"] = "-"
consortiums["ID"] = "-"

print(f"#Persons: {persons.shape[0]}")
print(f"#Organizations: {organizations.shape[0]}")
print(f"#Consortiums: {consortiums.shape[0]}")

#Persons: 20
#Organizations: 3
#Consortiums: 0


### Register agents

In [6]:
check_only = True  # Set to True to check without actually registering

#### Persons

In [21]:
for i, person in persons.iterrows():
    # Skip if already registered
    existing = client.search_entity(entity_type=models.Person, query={"pref_label": person["pref_label"]}).all()
    if len(existing) > 0:
        print(f"WARNING: Person '{person['pref_label']}' already registered - SKIPPING!")
        continue

    # Create model
    person_model = models.Person(
        pref_label=person["pref_label"],
        family_name=person["family_name"],
        given_name=person["given_name"],
    )

    # Register new entity
    if check_only:
        print(f"CHECK ONLY mode - Would register '{person['pref_label']}': {person_model}")
    else:
        registered_person = client.register_entity(person_model)
        print(f"Successfully registered person: {registered_person}")

        # Save in table
        persons.loc[i, "ID"] = registered_person.id

CHECK ONLY mode - Would register 'Reza Abbasi-Asl': id=None creation_date=None update_date=None created_by=None updated_by=None type='person' pref_label='Reza Abbasi-Asl' given_name='Reza' family_name='Abbasi-Asl' sub_id=None
CHECK ONLY mode - Would register 'Yazan N. Billeh': id=None creation_date=None update_date=None created_by=None updated_by=None type='person' pref_label='Yazan N. Billeh' given_name='Yazan N.' family_name='Billeh' sub_id=None
CHECK ONLY mode - Would register 'Binghuang Cai': id=None creation_date=None update_date=None created_by=None updated_by=None type='person' pref_label='Binghuang Cai' given_name='Binghuang' family_name='Cai' sub_id=None
CHECK ONLY mode - Would register 'Kael Dai': id=None creation_date=None update_date=None created_by=None updated_by=None type='person' pref_label='Kael Dai' given_name='Kael' family_name='Dai' sub_id=None
CHECK ONLY mode - Would register 'Nathan W. Gouwens': id=None creation_date=None update_date=None created_by=None updated_b

In [11]:
persons

,given_name,family_name,pref_label,ID
0,Reza,Abbasi-Asl,Reza Abbasi-Asl,-
1,Anton,Arkhipov,Anton Arkhipov,-
2,Yazan N.,Billeh,Yazan N. Billeh,-
3,Binghuang,Cai,Binghuang Cai,-
4,Jean-Denis,Courcol,Jean-Denis Courcol,-
5,Kael,Dai,Kael Dai,-
6,Michael,Gevaert,Michael Gevaert,-
7,Nathan W.,Gouwens,Nathan W. Gouwens,-
8,Sergey L.,Gratiy,Sergey L. Gratiy,-
9,Ramakrishnan,Iyer,Ramakrishnan Iyer,-


In [ ]:
# Write CSV with registered IDs
csv_file = f"registered_persons_{project_context.environment.value}_" + str(datetime.now()).replace(" ", "_").replace(":", "-") + ".csv"
persons.to_csv(os.path.join(agents_root, csv_file))

#### Organizations

In [20]:
for i, organization in organizations.iterrows():
    # Skip if already registered
    existing = client.search_entity(entity_type=models.Organization, query={"pref_label": organization["pref_label"]}).all()
    if len(existing) > 0:
        print(f"WARNING: Organization '{organization['pref_label']}' already registered - SKIPPING!")
        continue

    # Create model
    org_model = models.Organization(
        pref_label=organization["pref_label"],
    )

    # Register new entity
    if check_only:
        print(f"CHECK ONLY mode - Would register '{organization['pref_label']}': {org_model}")
    else:
        registered_org = client.register_entity(org_model)
        print(f"Successfully registered organization: {registered_org}")

        # Save in table
        organizations.loc[i, "ID"] = registered_org.id

CHECK ONLY mode - Would register 'UCSF Weill Institute for Neurosciences, Department of Neurology, University of California, San Francisco, CA, USA': id=None creation_date=None update_date=None created_by=None updated_by=None type='organization' pref_label='UCSF Weill Institute for Neurosciences, Department of Neurology, University of California, San Francisco, CA, USA' alternative_name=None


In [22]:
organizations

,pref_label,ID
0,"Allen Institute for Brain Science, Seattle, WA...",-
1,"UCSF Weill Institute for Neurosciences, Depart...",-
2,"Open Brain Institute, Lausanne, Switzerland",-


In [ ]:
# Write CSV with registered IDs
csv_file = f"registered_organizations_{project_context.environment.value}_" + str(datetime.now()).replace(" ", "_").replace(":", "-") + ".csv"
organizations.to_csv(os.path.join(agents_root, csv_file))

#### Consortiums

In [24]:
for i, consortium in consortiums.iterrows():
    # Skip if already registered
    existing = client.search_entity(entity_type=models.Consortium, query={"pref_label": consortium["pref_label"]}).all()
    if len(existing) > 0:
        print(f"WARNING: Consortium '{consortium['pref_label']}' already registered - SKIPPING!")
        continue

    # Create model
    consort_model = models.Consortium(
        pref_label=consortium["pref_label"],
    )

    # Register new entity
    if check_only:
        print(f"CHECK ONLY mode - Would register '{consortium['pref_label']}': {consort_model}")
    else:
        registered_consort = client.register_entity(consort_model)
        print(f"Successfully registered consortium: {registered_consort}")

        # Save in table
        consortiums.loc[i, "ID"] = registered_consort.id

In [25]:
consortiums

,pref_label,ID


In [ ]:
# Write CSV with registered IDs
csv_file = f"registered_consortia_{project_context.environment.value}_" + str(datetime.now()).replace(" ", "_").replace(":", "-") + ".csv"
consortiums.to_csv(os.path.join(agents_root, csv_file))